# Practice Notebook: Ingest a Fake Slide Deck into SQLite

This notebook lets you practice a simple version of a pitch-deck ingestion pipeline:

1. Define a fake slide deck as Python data
2. Normalize it into relational tables
3. Insert it into a SQLite database
4. Run a few analysis queries

The schema is intentionally simple so you can modify it later for your startup readiness project.


## 1) Imports and setup

In [7]:
import sqlite3
from pathlib import Path
from pprint import pprint
import json

DB_PATH = Path("fake_pitch_deck.db")
if DB_PATH.exists():
    DB_PATH.unlink()  # start fresh each run

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

print(f"Using database: {DB_PATH.resolve()}")

Using database: /Users/selenacooper/Desktop/startup-readiness-mvp/fake_pitch_deck.db


## 2) Create a fake slide deck

This simulates what a parser might produce after extracting content from a PDF or PPTX.


In [8]:
fake_deck = {
    "deck_id": "aetherfleet_deck",
    "company_name": "AetherFleet",
    "sector": "Logistics SaaS",
    "stage": "Seed",
    "team": "2 founders",
    "slides": [
        {
            "slide_id": "AetherFleet",
            "slide_number": 1,
            "slide_section": "Title",
            "extracted_text": "AetherFleet. AI dispatch and route optimization for regional carriers. Reduce empty miles by 18%. Deploy in less than 2 weeks.",
            "graphic_path": "graphics/slide_001_graphic.png"
        },
        {
            "slide_id": "The Issue People Face",
            "slide_number": 2,
            "slide_section": "Problem",
            "extracted_text": "Regional carriers still dispatch with spreadsheets, texts, and tribal knowledge. Low route visibility. Manual load assignment. High driver idle time.",
            "graphic_path": None
        },
        {
            "slide_id": "Current Users",
            "slide_number": 3,
            "slide_section": "Market",
            "extracted_text": "Large underserved mid-market freight segment. TAM: $9.2B. SAM: $1.4B. SOM: $120M.",
            "graphic_path": "graphics/slide_003_graphic.png"
        },
        {
            "slide_id": "Demonstrated Interest",
            "slide_number": 4,
            "slide_section": "Traction",
            "extracted_text": "Early pilots with paying fleets show strong retention. 12 paying fleets. $28K MRR. 109% net revenue retention.",
            "graphic_path": "graphics/slide_004_graphic.png"
        },
        {
            "slide_id": "AetherFleet Model",
            "slide_number": 5,
            "slide_section": "Business Model",
            "extracted_text": "Subscription pricing based on number of trucks. $199 per truck per month. Annual contracts. Setup fee for onboarding.",
            "graphic_path": None
        }
    ],
    "scores": [
        {"score_id": "score_1", "rubric_section": "Market Size", "value": 3.5},
        {"score_id": "score_2", "rubric_section": "Traction", "value": 4.5},
        {"score_id": "score_3", "rubric_section": "Team", "value": 3.0},
        {"score_id": "score_4", "rubric_section": "Business Model", "value": 4.0}
    ],
    "red_flags": [
        {"red_flag_id": "rf_1", "slide_id": "Current Users", "flag_type": "Unsubstantiated market size claim"},
        {"red_flag_id": "rf_2", "slide_id": None, "flag_type": "No mention of competition"}
    ]
}
pprint(fake_deck)

{'company_name': 'AetherFleet',
 'deck_id': 'aetherfleet_deck',
 'red_flags': [{'flag_type': 'Unsubstantiated market size claim',
                'red_flag_id': 'rf_1',
                'slide_id': 'Current Users'},
               {'flag_type': 'No mention of competition',
                'red_flag_id': 'rf_2',
                'slide_id': None}],
 'scores': [{'rubric_section': 'Market Size',
             'score_id': 'score_1',
             'value': 3.5},
            {'rubric_section': 'Traction', 'score_id': 'score_2', 'value': 4.5},
            {'rubric_section': 'Team', 'score_id': 'score_3', 'value': 3.0},
            {'rubric_section': 'Business Model',
             'score_id': 'score_4',
             'value': 4.0}],
 'sector': 'Logistics SaaS',
 'slides': [{'extracted_text': 'AetherFleet. AI dispatch and route '
                               'optimization for regional carriers. Reduce '
                               'empty miles by 18%. Deploy in less than 2 '
                   

## 3) Create SQLite schema

We will use four tables:

- `decks`: one row per pitch deck
- `slides`: one row per slide
- `bullets`: one row per bullet point
- `claims`: one row per extracted claim


In [9]:
schema_sql = '''
CREATE TABLE DECK (
    deck_id TEXT PRIMARY KEY,
    company_name TEXT NOT NULL,
    sector TEXT,
    stage TEXT,
    team TEXT
);

CREATE TABLE SLIDE (
    slide_id TEXT PRIMARY KEY,
    deck_id TEXT NOT NULL,
    slide_number INTEGER NOT NULL,
    slide_section TEXT,
    extracted_text TEXT,
    graphic_path TEXT,
    FOREIGN KEY (deck_id) REFERENCES DECK(deck_id) ON DELETE CASCADE
);

CREATE TABLE SCORE (
    score_id TEXT PRIMARY KEY,
    deck_id TEXT NOT NULL,
    rubric_section TEXT,
    value REAL,
    FOREIGN KEY (deck_id) REFERENCES DECK(deck_id) ON DELETE CASCADE
);

CREATE TABLE RED_FLAG (
    red_flag_id TEXT PRIMARY KEY,
    deck_id TEXT NOT NULL,
    slide_id TEXT,
    flag_type TEXT,
    FOREIGN KEY (deck_id) REFERENCES DECK(deck_id) ON DELETE CASCADE,
    FOREIGN KEY (slide_id) REFERENCES SLIDE(slide_id)
);
'''
cur.executescript(schema_sql)
conn.commit()

print("Schema created.")

Schema created.


## 4) Insert the fake deck into the database

In [ ]:
def _make_deck_id(evaluation):
    deck = evaluation.deck
    base = deck.company if deck and deck.company else deck.title if deck else "deck"
    return "".join(ch.lower() if ch.isalnum() else "_" for ch in base).strip("_") or "deck"


def ingest_deck(conn, evaluation):
    if evaluation.deck is None:
        raise ValueError("PitchDeckEvaluation.deck is required for ingestion")

    cur = conn.cursor()
    deck = evaluation.deck
    deck_id = _make_deck_id(evaluation)

    cur.execute(
        '''
        INSERT INTO DECK (deck_id, company_name, sector, stage, team)
        VALUES (?, ?, ?, ?, ?)
        ''',
        (
            deck_id,
            deck.company,
            deck.sector,
            deck.stage,
            json.dumps(deck.team, ensure_ascii=False),
        )
    )

    for slide in deck.slides:
        cur.execute(
            '''
            INSERT INTO SLIDE (slide_id, deck_id, slide_number, slide_section, extracted_text, graphic_path)
            VALUES (?, ?, ?, ?, ?, ?)
            ''',
            (
                f"{deck_id}_slide_{slide.slide_number}",
                deck_id,
                slide.slide_number,
                slide.section,
                slide.text,
                json.dumps(slide.graph_desc, ensure_ascii=False),
            )
        )

    rubric_sections = [
        ("Problem", evaluation.s1_problem),
        ("Solution", evaluation.s2_solution),
        ("Market Size", evaluation.s3_market_size),
        ("Product & Tech", evaluation.s4_product_and_tech),
        ("Business Model", evaluation.s5_business_model),
        ("Go-To-Market", evaluation.s6_go_to_market),
        ("Competition", evaluation.s7_competition),
        ("Team", evaluation.s8_team),
        ("Traction & KPIs", evaluation.s9_traction_and_kpis),
        ("The Ask & Financials", evaluation.s10_the_ask_and_financials),
    ]

    for index, (rubric_section, section) in enumerate(rubric_sections, start=1):
        cur.execute(
            '''
            INSERT INTO SCORE (score_id, deck_id, rubric_section, value)
            VALUES (?, ?, ?, ?)
            ''',
            (
                f"{deck_id}_score_{index}",
                deck_id,
                rubric_section,
                section.score if section.is_present else None,
            )
        )

    for index, red_flag in enumerate(evaluation.red_flags, start=1):
        cur.execute(
            '''
            INSERT INTO RED_FLAG (red_flag_id, deck_id, slide_id, flag_type)
            VALUES (?, ?, ?, ?)
            ''',
            (
                f"{deck_id}_red_flag_{index}",
                deck_id,
                None,
                red_flag,
            )
        )

    conn.commit()

# Example usage:
# evaluation = parser.evaluate_pitch_deck("sample_startup_deck.pdf")
# ingest_deck(conn, evaluation)
# print("PitchDeckEvaluation ingested.")

Fake deck ingested.


## 5) Inspect the tables

In [11]:
for table in ["DECK", "SLIDE", "SCORE", "RED_FLAG"]:
    count = cur.execute(f"SELECT COUNT(*) AS n FROM {table}").fetchone()["n"]
    print(f"{table}: {count} rows")

DECK: 1 rows
SLIDE: 5 rows
SCORE: 4 rows
RED_FLAG: 2 rows


In [12]:
rows = cur.execute("SELECT * FROM SLIDE ORDER BY slide_number").fetchall()
for row in rows:
    print(dict(row))

{'slide_id': 'AetherFleet', 'deck_id': 'aetherfleet_deck', 'slide_number': 1, 'slide_section': 'Title', 'extracted_text': 'AetherFleet. AI dispatch and route optimization for regional carriers. Reduce empty miles by 18%. Deploy in less than 2 weeks.', 'graphic_path': 'graphics/slide_001_graphic.png'}
{'slide_id': 'The Issue People Face', 'deck_id': 'aetherfleet_deck', 'slide_number': 2, 'slide_section': 'Problem', 'extracted_text': 'Regional carriers still dispatch with spreadsheets, texts, and tribal knowledge. Low route visibility. Manual load assignment. High driver idle time.', 'graphic_path': None}
{'slide_id': 'Current Users', 'deck_id': 'aetherfleet_deck', 'slide_number': 3, 'slide_section': 'Market', 'extracted_text': 'Large underserved mid-market freight segment. TAM: $9.2B. SAM: $1.4B. SOM: $120M.', 'graphic_path': 'graphics/slide_003_graphic.png'}
{'slide_id': 'Demonstrated Interest', 'deck_id': 'aetherfleet_deck', 'slide_number': 4, 'slide_section': 'Traction', 'extracted_t

## 6) Example queries

These are the kinds of queries that become useful once many decks are stored.


In [13]:
query = '''
SELECT
    rf.red_flag_id,
    rf.flag_type,
    rf.slide_id,
    s.slide_section,
    s.extracted_text,
    s.graphic_path
FROM RED_FLAG rf
LEFT JOIN SLIDE s ON rf.slide_id = s.slide_id
ORDER BY rf.red_flag_id
'''
rows = cur.execute(query).fetchall()
for row in rows:
    print(dict(row))

{'red_flag_id': 'rf_1', 'flag_type': 'Unsubstantiated market size claim', 'slide_id': 'Current Users', 'slide_section': 'Market', 'extracted_text': 'Large underserved mid-market freight segment. TAM: $9.2B. SAM: $1.4B. SOM: $120M.', 'graphic_path': 'graphics/slide_003_graphic.png'}
{'red_flag_id': 'rf_2', 'flag_type': 'No mention of competition', 'slide_id': None, 'slide_section': None, 'extracted_text': None, 'graphic_path': None}


In [14]:
query = '''
SELECT
    rubric_section,
    ROUND(value, 2) AS score
FROM SCORE
WHERE deck_id = 'aetherfleet_deck'
ORDER BY value DESC
'''
rows = cur.execute(query).fetchall()
for row in rows:
    print(dict(row))

{'rubric_section': 'Traction', 'score': 4.5}
{'rubric_section': 'Business Model', 'score': 4.0}
{'rubric_section': 'Market Size', 'score': 3.5}
{'rubric_section': 'Team', 'score': 3.0}


## 7) Export the database contents as JSON (optional)

This is useful if you want to inspect what ended up in the relational structure.


In [15]:
export = {
    "DECK": [dict(r) for r in cur.execute("SELECT * FROM DECK").fetchall()],
    "SLIDE": [dict(r) for r in cur.execute("SELECT * FROM SLIDE ORDER BY slide_number").fetchall()],
    "SCORE": [dict(r) for r in cur.execute("SELECT * FROM SCORE ORDER BY deck_id").fetchall()],
    "RED_FLAG": [dict(r) for r in cur.execute("SELECT * FROM RED_FLAG ORDER BY deck_id").fetchall()],
}

with open("deck_export.json", "w") as f:
    json.dump(export, f, indent=2)

print("Wrote deck_export.json")

Wrote deck_export.json


## 8) Practice extensions

Try any of these next:

1. Add a `scores` table for rubric labels
2. Add a `red_flags` table
3. Store per-slide embeddings metadata fields
4. Ingest multiple fake decks and compare sectors
5. Add a `raw_text` column for whole-slide text extraction
6. Build a `core10_section` normalization table


In [16]:
conn.close()
print("Connection closed.")

Connection closed.
